# iFinD HTTP API — 全市场 ETF 后复权行情抓取

用同花顺 iFinD HTTP API（`cmd_history_quotation` 历史行情函数）批量抓取**全市场交易型 ETF**的日线行情，
一次性把"够用 + 富余"的字段都拉下来，落地到 SQLite + Parquet，供因子策略使用。

**抓取内容（每只 ETF、每个交易日）**
- **原始行情（CPS=1，不复权）**：`open/high/low/close/preClose/avgPrice/change/changeRatio/volume/amount/turnoverRatio/transactionAmount`
  —— 这就是实盘真正成交的**二级市场价格**，用于冲击成本/流动性分析。
- **后复权行情（CPS=3，后复权·分红再投）**：`open/high/low/close` —— iFinD **原生后复权**价（非自算），用于回测做 total-return 序列。
- **基金专用**：`netAssetValue`(单位净值) / `accumulatedNAV`(累计净值) / `adjustedNAV`(复权净值) / `premium`(贴水) / `premiumRatio`(贴水率) —— 用于**折溢价/IOPV 偏离**过滤。

> **关于"后复权"**：iFinD 没有单独的后复权端点，它就是历史行情函数的 `CPS=3` 复权参数（基准日默认=上市日），
> 返回的就是官方后复权价——本 notebook 已直接用它，无需自己拿复权因子去乘。
> 注意 `adjustedNAV`(复权单位净值) 是**净值口径**的复权，和 `close(CPS=3)` 的**市价**后复权是两回事，两者都拉了。
> （`ths_af_stock` 复权因子是股票专用，对基金返回空，故未纳入。）

**关键限制（来自手册，已在代码中处理）**
- `access_token` 由 `refresh_token` 换取，**7 天有效、最多绑 20 个 IP**；过期错误码 `-1302`，自动重换。
- 每分钟最多 **600 条**请求（`-4400`）；单条历史行情命令最多 **200 万**数据点（`-4312`）→ 用 `CODES_PER_REQUEST` 控批。
- `-4102` 服务端超时是常见瞬时错误 → 自动退避重试。
- 周度报价额度上限（`-4302`，1.5 亿/周）→ 触发会**安全停止**并可断点续跑。


## 1. 配置
`REFRESH_TOKEN` 已填入你提供的 token（有效期至 **2026-07-15**，过期后到 iFinD 超级命令处更新）。

In [ ]:
# ====== 用户配置 ======
# 你的 refresh_token（有效期至 2026-07-15；过期后在 iFinD 超级命令/网页版更新）
REFRESH_TOKEN = "PASTE_YOUR_REFRESH_TOKEN_HERE"

# 数据范围
START_DATE = "2010-01-01"     # 起始（ETF 上市前的日期自动忽略）
END_DATE   = "2026-06-12"     # 截止

# 路径
UNIVERSE_JSON = "/Users/shenboheng/Documents/ClaudeCode/dataset/基金深度分析/bulk_universe.json"
OUT_DIR  = "/Users/shenboheng/Documents/ClaudeCode/dataset/基金深度分析"
DB_PATH  = OUT_DIR + "/etf_market_ifind.db"
PARQUET  = OUT_DIR + "/etf_market_ifind.parquet"

# 抓取字段
RAW_INDICATORS = ["open","high","low","close","preClose","avgPrice","change","changeRatio",
                  "volume","amount","turnoverRatio","transactionAmount",
                  "netAssetValue","accumulatedNAV","adjustedNAV","premium","premiumRatio"]
HFQ_INDICATORS = ["open","high","low","close"]   # 后复权(CPS=3)，回测用
CPS_RAW, CPS_HFQ = "1", "3"                       # 1=不复权  3=后复权(分红再投)

# 批量 / 限流
CODES_PER_REQUEST = 12     # 单条命令的 ETF 数（×字段×天数 须 < 200万）
REQUESTS_PER_MIN  = 500    # 留余量，手册上限 600/min
HTTP_TIMEOUT      = 180

MIN_INTERVAL = 60.0 / REQUESTS_PER_MIN
print("min interval per request:", round(MIN_INTERVAL, 3), "s")


## 2. Token 管理
`refresh_access_token` 用 `refresh_token` 换当前有效的 `access_token`；`-1302` 失效时自动重换。

In [ ]:
import requests, json, time

BASE = "https://quantapi.51ifind.com/api/v1"
_state = {"token": None, "last_req": 0.0}

class QuotaStop(Exception):
    """周/月额度耗尽，安全停止。"""

def refresh_access_token():
    r = requests.post(f"{BASE}/get_access_token",
                      headers={"Content-Type": "application/json", "refresh_token": REFRESH_TOKEN},
                      timeout=60)
    j = r.json()
    tok = (j.get("data") or {}).get("access_token")
    if not tok:
        raise RuntimeError(f"获取 access_token 失败: {j}")
    _state["token"] = tok
    print("access_token OK:", tok[:12], "...")
    return tok

def _throttle():
    dt = time.time() - _state["last_req"]
    if dt < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - dt)
    _state["last_req"] = time.time()

refresh_access_token()


## 3. 历史行情请求（带重试 / 限流 / 错误处理）

In [ ]:
QUOTA_CODES = {-4301, -4302, -4303, -4317, -4318}

def history_quotation(codes, indicators, cps, start=START_DATE, end=END_DATE, _depth=0):
    """调用 cmd_history_quotation，返回原始 JSON。codes: list[str] iFinD thscode。"""
    if _state["token"] is None:
        refresh_access_token()
    payload = {
        "codes": ",".join(codes),
        "indicators": ",".join(indicators),
        "startdate": start,
        "enddate": end,
        # CPS 复权方式; Fill=Blank 非交易日留空; BaseDate 空 -> 后复权按上市日(手册默认)
        "functionpara": {"CPS": cps, "Fill": "Blank"},
    }
    headers = {"Content-Type": "application/json", "access_token": _state["token"]}
    _throttle()
    try:
        r = requests.post(f"{BASE}/cmd_history_quotation", json=payload, headers=headers, timeout=HTTP_TIMEOUT)
        j = r.json()
    except Exception:
        if _depth >= 5:
            raise
        time.sleep(5 * (_depth + 1))
        return history_quotation(codes, indicators, cps, start, end, _depth + 1)

    ec = j.get("errorcode", j.get("errcode", 0))
    if ec in (0, None):
        return j
    if ec == -4001:                            # 无数据
        return {"tables": []}
    if ec in (-1300, -1302) and _depth < 3:    # token 失效 -> 重换重试
        refresh_access_token()
        return history_quotation(codes, indicators, cps, start, end, _depth + 1)
    if ec in (-4102, -4101, -5002, -5004) and _depth < 6:   # 服务端瞬时错误/超时 -> 退避
        time.sleep(6 * (_depth + 1))
        return history_quotation(codes, indicators, cps, start, end, _depth + 1)
    if ec == -4400 and _depth < 6:             # 每分钟限流 -> 退避
        time.sleep(20)
        return history_quotation(codes, indicators, cps, start, end, _depth + 1)
    if ec in QUOTA_CODES:                       # 额度耗尽 -> 安全停止
        raise QuotaStop(f"{ec}: {j.get('errmsg')}")
    if _depth < 4:
        time.sleep(5 * (_depth + 1))
        return history_quotation(codes, indicators, cps, start, end, _depth + 1)
    print(f"  ✗ 放弃 batch (ec={ec}): {j.get('errmsg')} | codes={codes[:3]}...")
    return {"tables": []}


import pandas as pd

def parse_tables(j, suffix=""):
    """把 iFinD tables 结构解析成长表 (thscode,date,字段...)。"""
    frames = []
    for t in (j.get("tables") or []):
        code = t.get("thscode")
        times = t.get("time") or []
        table = t.get("table") or {}
        if not times:
            continue
        d = {"thscode": code, "date": times}
        for k, v in table.items():
            v = list(v)
            if len(v) < len(times):
                v = v + [None] * (len(times) - len(v))
            d[k + suffix] = v[:len(times)]
        frames.append(pd.DataFrame(d))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


## 4. 构建 ETF 代码表（仅交易型 ETF）

从你现有的 `bulk_universe.json` 取"名字含 ETF/交易型开放式"的产品，映射到 iFinD thscode：
`5xxxxx → .SH`、`1xxxxx → .SZ`；剔除联接、ETF-FOF、以及 `0` 开头的场外 OF（无二级市场价）。

In [ ]:
from pathlib import Path

raw_uni = json.loads(Path(UNIVERSE_JSON).read_text(encoding="utf-8"))["data"]

def to_thscode(code: str):
    if code.startswith("5"):
        return code + ".SH"
    if code.startswith("1"):
        return code + ".SZ"
    return None     # 0 开头多为场外 OF / 货币 C 类 / FOF

rows = []
for code, item in raw_uni.items():
    name = str(item.get("name", ""))
    up = name.upper()
    if not (("ETF" in up) or ("交易型开放式" in name)):
        continue
    if ("联接" in name) or ("FOF" in up):
        continue
    ths = to_thscode(code)
    if ths is None:
        continue
    rows.append({"fund_code": code, "thscode": ths, "name": name})

codes_df = (pd.DataFrame(rows).drop_duplicates("thscode")
            .sort_values("thscode").reset_index(drop=True))
print(f"交易型 ETF 代码数: {len(codes_df)}")

# 如需手动指定/缩小范围，取消下一行注释自定义：
# codes_df = codes_df[codes_df["thscode"].isin(["510300.SH","159915.SZ","518880.SH","511880.SH"])]

codes_df.head(10)


## 5. 小样本自测（先验证字段都通，再跑全量）

In [ ]:
test_codes = ["510300.SH", "159915.SZ", "518880.SH", "511880.SH"]   # 沪深300/创业板/黄金/银华日利
_raw = parse_tables(history_quotation(test_codes, RAW_INDICATORS, CPS_RAW), suffix="")
_hfq = parse_tables(history_quotation(test_codes, HFQ_INDICATORS, CPS_HFQ), suffix="_hfq")
print("raw rows:", len(_raw), "| hfq rows:", len(_hfq), "| 字段:", [c for c in _raw.columns if c not in ('thscode','date')])
chk = _raw.merge(_hfq, on=["thscode", "date"], how="inner")
# 后复权 close 与 原始 close 的累计差异 = 累计分红效应（无分红的如黄金ETF两者相等）
if {"close", "close_hfq"} <= set(chk.columns):
    g = chk.dropna(subset=["close","close_hfq"]).groupby("thscode")
    diff = (g.apply(lambda x: float((x["close_hfq"]/x["close"]).iloc[-1] - (x["close_hfq"]/x["close"]).iloc[0]),
                    include_groups=False))
    print("\\n各 ETF 后复权/原始 比值的首末变化(≈累计分红再投影响，0=期间无分红):")
    print(diff.round(4).to_string())
_raw.head(3)


## 6. 全量抓取（分批、可断点续跑）

每批同时取**原始**(CPS=1)与**后复权**(CPS=3)，按 `(thscode,date)` 合并后追加写入 SQLite。
已写入的 `thscode` 会跳过，可随时中断后重跑续传。

In [ ]:
import sqlite3
TABLE = "etf_quote"

def existing_codes(db_path):
    try:
        con = sqlite3.connect(db_path)
        try:
            return set(pd.read_sql_query(f"SELECT DISTINCT thscode FROM {TABLE}", con)["thscode"])
        finally:
            con.close()
    except Exception:
        return set()

def chunks(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i + n]

def fetch_batch(batch):
    raw = parse_tables(history_quotation(batch, RAW_INDICATORS, CPS_RAW), suffix="")
    hfq = parse_tables(history_quotation(batch, HFQ_INDICATORS, CPS_HFQ), suffix="_hfq")
    if raw.empty:
        return hfq
    if hfq.empty:
        return raw
    return raw.merge(hfq, on=["thscode", "date"], how="left")

def run_full(resume=True):
    all_codes = codes_df["thscode"].tolist()
    done = existing_codes(DB_PATH) if resume else set()
    todo = [c for c in all_codes if c not in done]
    print(f"待抓 {len(todo)} / 共 {len(all_codes)} 只（已完成 {len(done)} 只）")
    con = sqlite3.connect(DB_PATH)
    written = 0
    try:
        for bi, batch in enumerate(chunks(todo, CODES_PER_REQUEST), 1):
            try:
                df = fetch_batch(batch)
            except QuotaStop as e:
                print(f"⏸ 额度耗尽，已安全停止：{e}\\n   稍后重跑本 cell 即从断点续传。")
                break
            if df is not None and not df.empty:
                for c in df.columns:
                    if c not in ("thscode", "date"):
                        df[c] = pd.to_numeric(df[c], errors="coerce")
                df["fund_code"] = df["thscode"].str.split(".").str[0]
                df.to_sql(TABLE, con, if_exists="append", index=False)
                written += df["thscode"].nunique()
            got = set(df["thscode"]) if (df is not None and not df.empty) else set()
            missed = [c for c in batch if c not in got]
            print(f"[{bi}] {batch[0]}..{batch[-1]}  写入 {0 if df is None else len(df)} 行"
                  + (f"  | 无数据 {len(missed)}" if missed else ""))
    finally:
        con.close()
    print(f"\\n✅ 本轮新增 {written} 只 ETF -> {DB_PATH}")

run_full(resume=True)


## 7. 建索引 + 导出 Parquet + 质检

In [ ]:
con = sqlite3.connect(DB_PATH)
try:
    con.execute(f"CREATE INDEX IF NOT EXISTS idx_code_date ON {TABLE}(thscode, date)")
    con.commit()
    full = pd.read_sql_query(f"SELECT * FROM {TABLE}", con)
finally:
    con.close()

full["date"] = pd.to_datetime(full["date"])
full = full.drop_duplicates(["thscode", "date"]).sort_values(["thscode", "date"])
full.to_parquet(PARQUET, index=False)
print("总行数:", len(full), "| ETF 数:", full["thscode"].nunique(),
      "| 日期:", full["date"].min().date(), "->", full["date"].max().date())

cover = (full.groupby("thscode")
         .agg(rows=("date", "size"), start=("date", "min"), end=("date", "max"))
         .sort_values("rows"))
print("\\n最短历史的 5 只:"); display(cover.head())

if "premiumRatio" in full.columns:
    pr = full["premiumRatio"].dropna()
    print(f"\\n贴水率分布: 中位 {pr.median():.3f}%  P95 {pr.quantile(.95):.3f}%  P99 {pr.quantile(.99):.3f}%  max {pr.abs().max():.2f}%")
    print("提示: |溢价率| 高的多为 QDII/商品 ETF，实盘前据此设折溢价过滤上限。")

if "amount" in full.columns:
    adv = full[full["date"] >= full["date"].max() - pd.Timedelta(days=30)].groupby("thscode")["amount"].mean()
    print(f"\\n近月日均成交额 < 3000万 的 ETF 数: {(adv < 3e7).sum()} / {adv.notna().sum()}（实盘流动性过滤候选）")


## 8. 接入策略（下一步）

落地的 `etf_market_ifind.db / .parquet` 含每只 ETF：市价 OHLCV、后复权 OHLC、单位/累计/复权净值、成交额、贴水率。建议：
1. **用后复权 close 重建一版回测**（替代累计净值口径），对比 NAV 口径差多少 → 量化"净值 vs 市价"缺口。
2. **流动性过滤**：`amount` 近 20 日均值下限（如 ≥3000 万–1 亿）。
3. **折溢价过滤**：`premiumRatio` 绝对值上限，重点限制 QDII/商品 ETF。
4. **冲击成本**：用 `amount` 估 ADV，按下单额/ADV 设非线性冲击成本，替代固定 bps。
